# 01 — Baseline за 30 минут
**Цель первого часа:** любая валидная метрика + первый сабмит. Улучшать будем потом.

1. Заполни конфиг ниже  2. Run All  3. Запиши метрику в чат команды

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd
pd.set_option("display.max_columns", 100)
from src.utils import read_table, reduce_mem_usage, seed_everything
from src.features import TablePrep
from src.train import run, detect_task
from src.metrics import score, best_threshold
seed_everything(42)

## ⚙️ Конфиг — единственное, что нужно поменять

In [ ]:
TRAIN = ROOT / "data/raw/train.csv"
TEST  = ROOT / "data/raw/test.csv"      # None, если теста нет
TARGET = "target"
ID_COL = "id"                            # None, если нет
METRIC = None                            # None = авто (roc_auc / f1_macro / rmse) или "gini", "f1", "rmse"...
GROUP_COL = None                         # например "client_id", если у клиента несколько строк
TIME_COL = None                          # например "report_date" -> валидация по времени
MODELS = "lgb,cb,xgb"                    # недоступные пропустятся автоматически
FOLDS = 5

## 1. Быстрый взгляд на данные
Полный отчёт: `python eda/quick_eda.py data/raw/train.csv --target target --test data/raw/test.csv`

In [ ]:
train = reduce_mem_usage(read_table(TRAIN))
test = read_table(TEST) if TEST else None
print(train.shape, None if test is None else test.shape)
print("Задача:", detect_task(train[TARGET]))
display(train.head())
display(train[TARGET].value_counts(normalize=True).head(10))

In [ ]:
# Пропуски и типы
pd.DataFrame({"dtype": train.dtypes, "na_%": train.isna().mean().mul(100).round(1), "nunique": train.nunique()}).sort_values("na_%", ascending=False)

## 2. Baseline: CV + бленд + сабмит + бандл для API

In [ ]:
bundle, oof, scores = run(TRAIN, TARGET, TEST, ID_COL, models=MODELS, folds=FOLDS, metric=METRIC,
                          group=GROUP_COL, time_col=TIME_COL, tag="nb_baseline")
scores

## 3. Важность признаков и анализ ошибок

In [ ]:
fi = pd.read_csv(ROOT / "reports/feature_importance.csv", index_col=0) if (ROOT / "reports/feature_importance.csv").exists() else None
if fi is not None:
    fi.head(25).plot.barh(figsize=(7, 8)).invert_yaxis()

In [ ]:
# Где модель ошибается сильнее всего? (смотрим глазами — отсюда рождаются новые фичи)
y_true = train[TARGET]
pred = np.mean(list(oof.values()), axis=0)
if pred.ndim == 1 and bundle["task"] != "regression":
    err = train.assign(pred=pred, err=np.abs((y_true == bundle["classes"][1]).astype(int) - pred))
elif bundle["task"] == "regression":
    err = train.assign(pred=pred, err=np.abs(y_true - pred))
else:
    err = train.assign(pred=pred.argmax(1))
    err["err"] = (err["pred"] != y_true.map({c: i for i, c in enumerate(bundle["classes"])})).astype(int)
err.sort_values("err", ascending=False).head(20)

## 4. Журнал экспериментов
Каждый запуск `run()` дописывает строку в `reports/experiments.csv`. Меняйте `tag`, чтобы помнить, что пробовали.

In [ ]:
pd.read_csv(ROOT / "reports/experiments.csv").tail(10)

## 5. Что дальше (по приоритету)
1. **Фичи** в `src/features.py → add_features` (отношения, агрегаты по клиенту, даты) — даёт больше всего
2. Проверить утечки (фичи из будущего, id, коррелирующие с таргетом на 0.99)
3. Подобрать параметры (Optuna, 30–50 trials, только лучшей модели)
4. Бленд разных моделей / сидов
5. Не забыть: демо (Streamlit) и презентация важнее +0.002 к метрике